# 02 — BCRD CPI & Family Basket | Bronze Ingestion

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Ingest the six official BCRD price-index workbooks into Bronze **exactly as published**, so that Silver can parse each layout with full lineage back to the source cell.

## Why row-level capture
These workbooks are presentation tables, not datasets: titles, multi-row headers, a year written once per block, Spanish month names, index/variation column pairs, and methodology footnotes. Each file has a different layout. Forcing them into typed columns at ingestion would mean making interpretation decisions in Bronze.

Bronze therefore stores **every row of every sheet** as an array of text cells, plus lineage metadata. Nothing is interpreted, dropped or corrected here — including the methodology notes, which Silver keeps as documented context.

## Sources (raw landing `/Volumes/rd_economic_intelligence/bronze/bcrd_raw/cpi/`)
| dataset_code | File | Content |
|---|---|---|
| `cpi_national` | `ipc_base_2019-2020.xlsx` | National CPI index and variations, 1984 onward |
| `cpi_groups` | `ipc_grupos_base_2019-2020.xlsx` | CPI by 12 expenditure groups, 1999 onward |
| `cpi_core` | `ipc_subyacente_base_2019-2020.xlsx` | Core CPI, 2000 onward |
| `cpi_regions` | `ipc_regiones_base_2019-2020.xlsx` | CPI by region, 2011 onward |
| `basket_quintiles` | `Costo_Canasta_quintiles_base_2019-2020.xlsx` | Family basket cost by income quintile and national, 2018 onward |
| `basket_regions` | `Costo_Canasta_regiones_base_2019-2020.xlsx` | Family basket cost by region, Oct-2020 onward |

**Target:** `rd_economic_intelligence.bronze.bcrd_cpi_raw_rows` — idempotent per source file.

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
# Step 0 — Dependencies: serverless does not ship openpyxl, required to read .xlsx
dbutils.library.restartPython()

In [0]:
# ============================================================
# Step 1 — Configuration
# ============================================================
from datetime import datetime, timezone
import uuid

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/bcrd_raw/cpi/"
TARGET_TABLE = f"{CATALOG}.bronze.bcrd_cpi_raw_rows"

# dataset_code -> (file name, text that must appear in the header area: structural contract)
SOURCES = {
    "cpi_national":     ("ipc_base_2019-2020.xlsx",                     ["Período", "Indice"]),
    "cpi_groups":       ("ipc_grupos_base_2019-2020.xlsx",              ["Período", "Alimentos y Bebidas"]),
    "cpi_core":         ("ipc_subyacente_base_2019-2020.xlsx",          ["Período", "Subyacente"]),
    "cpi_regions":      ("ipc_regiones_base_2019-2020.xlsx",            ["Período", "Región Ozama"]),
    "basket_quintiles": ("Costo_Canasta_quintiles_base_2019-2020.xlsx", ["Período", "Quintil 1", "Nacional"]),
    "basket_regions":   ("Costo_Canasta_regiones_base_2019-2020.xlsx",  ["Período", "Región Ozama"]),
}
HEADER_SCAN_ROWS = 10

BATCH_ID = str(uuid.uuid4())
INGESTION_TS = datetime.now(timezone.utc)
print(f"Batch ID: {BATCH_ID}")

In [0]:
# ============================================================
# Step 2 — Raw landing verification
# ============================================================
landed = {f.name: f for f in dbutils.fs.ls(RAW_PATH)}
print("Files found:")
for f in landed.values():
    print(f"- {f.name} | {f.size:,} bytes")

missing = [file for file, _ in SOURCES.values() if file not in landed]
if missing:
    raise FileNotFoundError(f"Missing source files in {RAW_PATH}: {missing}")

In [0]:
# ============================================================
# Step 3 — Read every row of every sheet as text, with lineage
# ============================================================
import openpyxl


def as_text(value):
    """Text exactly as stored: floats keep full precision (repr), empty cells stay None."""
    if value is None:
        return None
    if isinstance(value, float):
        return repr(value)
    return str(value)


rows = []
contract_results = {}
for dataset_code, (file_name, required_tokens) in SOURCES.items():
    path = RAW_PATH + file_name
    info = landed[file_name]
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)

    header_text = ""
    for sheet_index, ws in enumerate(wb.worksheets):
        for row_number, values in enumerate(ws.iter_rows(values_only=True), start=1):
            cells = [as_text(v) for v in values]
            while cells and cells[-1] is None:
                cells.pop()
            if sheet_index == 0 and row_number <= HEADER_SCAN_ROWS:
                header_text += " ".join(c for c in cells if c) + " "
            rows.append((
                dataset_code, file_name, ws.title, sheet_index, row_number,
                cells, sum(1 for c in cells if c is not None),
                path, int(info.size),
                datetime.fromtimestamp(info.modificationTime / 1000, tz=timezone.utc),
            ))
    wb.close()

    # Structural contract: the expected header tokens must be present
    absent = [t for t in required_tokens if t not in header_text]
    contract_results[dataset_code] = absent
    print(f"{dataset_code:<17} {file_name:<45} contract {'PASS' if not absent else 'FAIL ' + str(absent)}")

failed = {k: v for k, v in contract_results.items() if v}
if failed:
    raise ValueError(f"Source layout changed (missing header tokens): {failed}")

In [0]:
# ============================================================
# Step 4 — Build the Bronze DataFrame
# ============================================================
from pyspark.sql import functions as F

schema = (
    "dataset_code string, _source_file string, _source_sheet string, _source_sheet_index int, "
    "_source_row_number int, cells array<string>, non_empty_cells int, "
    "_source_path string, _source_file_size_bytes bigint, _source_modified_ts timestamp"
)
df = (
    spark.createDataFrame(rows, schema)
    .withColumn("_ingestion_ts", F.lit(INGESTION_TS).cast("timestamp"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

summary = (df.groupBy("dataset_code", "_source_file")
             .agg(F.count("*").alias("rows"), F.sum(F.when(F.col("non_empty_cells") > 0, 1).otherwise(0)).alias("non_empty_rows"))
             .orderBy("dataset_code"))
display(summary)

In [0]:
# ============================================================
# Step 5 — Idempotent write (replace rows from the same files)
# ============================================================
files = [file for file, _ in SOURCES.values()]
if spark.catalog.tableExists(TARGET_TABLE):
    in_list = ", ".join(f"'{f}'" for f in files)
    spark.sql(f"DELETE FROM {TARGET_TABLE} WHERE _source_file IN ({in_list})")

df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)

spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Bronze: BCRD CPI and family-basket workbooks captured row by row as published (text cells), with lineage. Parsed into tidy tables in Silver.'
""")

In [0]:
# ============================================================
# Step 6 — Verification
# ============================================================
read_rows = len(rows)
loaded = spark.table(TARGET_TABLE).filter(F.col("_source_file").isin(files)).count()
assert loaded == read_rows, f"Row count mismatch: read {read_rows}, loaded {loaded}"
print(f"PASS: {loaded:,} rows loaded from {len(files)} files — matches rows read")

import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "batch_id": BATCH_ID,
    "rows_loaded": loaded,
    "per_dataset": {r["dataset_code"]: {"rows": r["rows"], "non_empty_rows": r["non_empty_rows"]} for r in summary.collect()},
}))